# 04 — Model Evaluation

**HemoVisionAI** — Acute Lymphoblastic Leukemia (ALL) Classification

Comprehensive evaluation of HemoVisionNet V1.
All configuration from `config.py`; data loading from `preprocessing.data_pipeline`.

---

In [ ]:
# ============================================================
# Cell 1: Imports and Project Setup
# ============================================================
import os
import sys
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import load_model
from sklearn.metrics import (
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_curve, auc,
)
from sklearn.preprocessing import label_binarize

# Add project root to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Central configuration
from config import (
    DATASET_PATH, IMAGE_SIZE, BATCH_SIZE, SEED, VAL_SPLIT,
    NUM_CLASSES, CHECKPOINT_PATH,
)

# Data pipeline
from preprocessing.data_pipeline import get_datasets

print(f"TensorFlow version: {tf.__version__}")
print(f"Checkpoint path:    {CHECKPOINT_PATH}")

## 1. Load Model

In [ ]:
# ============================================================
# Cell 2: Load Trained Model
# ============================================================
assert os.path.exists(CHECKPOINT_PATH), (
    f"Checkpoint not found: {CHECKPOINT_PATH}\n"
    "Run 03_model_development.ipynb first to train and save the model."
)

model = load_model(CHECKPOINT_PATH)

print(f"✅ Model loaded: {model.name}")
print(f"   Output shape: {model.output_shape}")
assert model.output_shape[-1] == NUM_CLASSES

# Detect Rescaling layer
has_rescaling = any(isinstance(l, tf.keras.layers.Rescaling) for l in model.layers)
print(f"   Rescaling layer present: {has_rescaling}")
if has_rescaling:
    print("   → Pipeline must deliver [0, 255] images (model normalises internally)")

## 2. Load Validation Data

Uses the **exact same** `get_datasets()` function and `random_state=42` as training,
guaranteeing identical splits.

In [ ]:
# ============================================================
# Cell 3: Load Validation Dataset (identical split to training)
# ============================================================
_, val_ds, class_names = get_datasets(
    dataset_dir=DATASET_PATH,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    seed=SEED,
    augment_fn=None,
)

assert len(class_names) == NUM_CLASSES

# Pixel range check
for images, _ in val_ds.take(1):
    pmax = tf.reduce_max(images).numpy()
    assert pmax > 1.0, (
        f"Pixel max is {pmax:.4f} — images are pre-normalised. "
        "Double-normalisation will make the model predict a single class."
    )
print("✅ Pixel range verified: [0, 255]")

## 3. Generate Predictions

In [ ]:
# ============================================================
# Cell 4: Predictions
# ============================================================
y_true      = np.concatenate([labels.numpy() for _, labels in val_ds])
y_pred_prob = model.predict(val_ds, verbose=1)
y_pred      = np.argmax(y_pred_prob, axis=1)

# Assertions
assert y_pred.shape == y_true.shape
assert y_pred_prob.shape[1] == NUM_CLASSES
assert set(np.unique(y_true)) == set(range(NUM_CLASSES)), (
    f"y_true has classes {set(np.unique(y_true))}, expected {set(range(NUM_CLASSES))}"
)

print(f"\ny_true unique:  {sorted(np.unique(y_true))}")
print(f"y_pred unique:  {sorted(np.unique(y_pred))}")

# Warn if single-class predictions
if len(np.unique(y_pred)) == 1:
    print("⚠️  CRITICAL: Model predicts only one class — normalisation mismatch!")

print(f"\nTrue label distribution:")
for i, name in enumerate(class_names):
    print(f"  [{i}] {name}: {np.sum(y_true == i)}")
print(f"\nPredicted label distribution:")
for i, name in enumerate(class_names):
    print(f"  [{i}] {name}: {np.sum(y_pred == i)}")

In [ ]:
# ============================================================
# Cell 5: Accuracy
# ============================================================
accuracy = np.mean(y_true == y_pred)
print(f"Overall Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

## 4. Confusion Matrix

In [ ]:
# ============================================================
# Cell 6: Confusion Matrix
# ============================================================
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor="gray", ax=ax)
ax.set_title("Confusion Matrix — HemoVisionNet V1", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
plt.xticks(rotation=20, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

## 5. Classification Report

In [ ]:
# ============================================================
# Cell 7: Classification Report
# ============================================================
print(classification_report(y_true, y_pred, target_names=class_names))

## 6. Per-Class Metrics

In [ ]:
# ============================================================
# Cell 8: Per-Class Precision / Recall / F1
# ============================================================
precision_per_class = precision_score(y_true, y_pred, average=None)
recall_per_class    = recall_score(y_true, y_pred, average=None)
f1_per_class        = f1_score(y_true, y_pred, average=None)

x = np.arange(NUM_CLASSES); width = 0.25
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, precision_per_class, width, label="Precision", color="#3498db")
ax.bar(x,          recall_per_class,    width, label="Recall",    color="#e74c3c")
ax.bar(x + width,  f1_per_class,        width, label="F1-Score",  color="#2ecc71")
ax.set_xticks(x); ax.set_xticklabels(class_names, rotation=15, ha="right")
ax.set_ylim(0, 1.15); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Per-Class Metrics", fontsize=14, fontweight="bold")
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=3)
plt.tight_layout(); plt.show()

## 7. ROC-AUC

In [ ]:
# ============================================================
# Cell 9: ROC-AUC Curves (One-vs-Rest)
# ============================================================
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12"]
auc_scores = []

fig, ax = plt.subplots(figsize=(10, 8))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr); auc_scores.append(roc_auc)
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f"{class_names[i]} (AUC={roc_auc:.4f})")
ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.5,label="Random (0.5)")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(loc="lower right")
ax.set_title("ROC Curves (OvR)", fontsize=14, fontweight="bold"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

macro_auc = np.mean(auc_scores)
print(f"\nMacro-Average AUC: {macro_auc:.4f}")

## 8. Misclassified Images

In [ ]:
# ============================================================
# Cell 10: Misclassified Samples
# ============================================================
misclassified = np.where(y_true != y_pred)[0]
print(f"Misclassified: {len(misclassified)} / {len(y_true)} ({len(misclassified)/len(y_true)*100:.2f}%)")

all_images = np.concatenate([img.numpy() for img, _ in val_ds])
num_show = min(16, len(misclassified))

if num_show == 0:
    print("✅ Perfect accuracy — no misclassified images!")
else:
    cols = 4; rows = (num_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = axes.flatten()
    for i in range(num_show):
        idx = misclassified[i]; ax = axes[i]
        ax.imshow(all_images[idx].astype(np.uint8))
        ax.set_title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}\n"
                     f"Conf: {y_pred_prob[idx][y_pred[idx]]:.2%}", fontsize=8, color="red", fontweight="bold")
        ax.axis("off")
    for j in range(num_show, len(axes)):
        axes[j].axis("off")
    plt.suptitle(f"Misclassified Images ({num_show} of {len(misclassified)})", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

## 9. Summary

In [ ]:
# ============================================================
# Cell 11: Final Summary
# ============================================================
print("=" * 60)
print("  HemoVisionNet V1 — Evaluation Summary")
print("=" * 60)
print(f"  Accuracy:           {accuracy:.4f}")
print(f"  Macro-Avg Precision: {np.mean(precision_per_class):.4f}")
print(f"  Macro-Avg Recall:    {np.mean(recall_per_class):.4f}")
print(f"  Macro-Avg F1:        {np.mean(f1_per_class):.4f}")
print(f"  Macro-Avg AUC:       {macro_auc:.4f}")
print(f"  Misclassified:       {len(misclassified)} / {len(y_true)}")
print("=" * 60)